<a href="https://colab.research.google.com/github/adam01see/Neural-network-from-scratch-with-NumPy/blob/main/NNwithNP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.datasets import fetch_openml

# 1. Download the MNIST dataset from the web
print("Downloading MNIST dataset (this might take a minute)...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

# 2. Combine the pixel features (X) and the text labels (y) into one big matrix
# We cast the labels to integers so they match Samson's code exactly
X_raw = mnist.data
Y_raw = mnist.target.astype(int)

# 3. Reconstruct the 'data' variable exactly like Samson's Kaggle format
# Stack Y as the first column, followed by the 784 columns of pixel data
data = np.hstack((Y_raw.reshape(-1, 1), X_raw))

In [12]:
m, n = data.shape
np.random.shuffle(data)
# shuffling data

data_dev = data[0:1000].T # testing set
Y_dev = data_dev[0] # labels
X_dev = data_dev[1:n] # pixel values 1-784
X_dev = X_dev / 255. # normalazing data (0-1)

data_train = data[1000:m].T # training set
Y_train = data_train[0] # labels
X_train = data_train[1:n] # pixel values 1-784
X_train = X_train / 255. # normalazing data (0-1)

In [17]:
# randomly initialising the weights and biases
def init_parameters():
  W1 = np.random.rand (10, 784) - 0.5
  b1 = np.random.rand (10, 1) - 0.5
  W2 = np.random.rand (10, 10) - 0.5
  b2 = np.random.rand (10, 1) - 0.5

  return W1, b1, W2, b2

#defining the ReLU activation function
def ReLU(Z):
  return np.maximum (0, Z)

#defining the Softmax function

def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)


def forward_propagation (W1, b1, W2, b2, X):
  """
  forwad prop. operations, applying weights and biases to both hidden layers
  as well as activation functions
  """
  Z1 = W1.dot(X) + b1
  A1 = ReLU(Z1)
  Z2 = W2.dot(A1) + b2
  A2 = softmax(Z2)
  return Z1, A1, Z2, A2


def one_hot(Y):
  """
  Converting the output to 10 demensional vector and assigning 1 to the
  the highest probability and 0 to the rest
  """
  one_hot_Y = np.zeros((Y.size, Y.max() +1))
  one_hot_Y[np.arange(Y.size), Y] = 1
  one_hot_Y = one_hot_Y.T
  return one_hot_Y

# derivative of the RelU function
def deriv_ReLU(Z):
  return Z > 0

def back_propagation (Z1, A1, Z2, A2, W2, X, Y):
  """
  calculating the error and calculating how should the weights be adjusted
  """
  m = Y.size
  one_hot_Y = one_hot(Y)
  dZ2 = A2 - one_hot_Y

  dW2 = 1 / m * dZ2.dot(A1.T)
  db2 =  1 / m * np.sum(dZ2)
  dZ1 = W2.T.dot(dZ2) * deriv_ReLU(Z1)

  dW1 = 1 / m * dZ1.dot(X.T)
  db1 =  1 / m * np.sum(dZ1)
  return dW1, db1, dW2, db2

def update_parameters(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
  """
  updating the parameters by multiplying the errors multiplied by the learning rate alpha
  """
  W1 = W1 - alpha * dW1
  b1 = b1 - alpha * db1
  W2 = W2 - alpha * dW2
  b2 = b2 - alpha * db2
  return W1, b1, W2, b2


In [29]:
#returns the prediction based on the strongest neuron
def get_predictions(A2):
  return np.argmax(A2, 0)

#calculates the accuracy of the model by deviding prediction with the actual label
def get_accuracy(predictions, Y):
  print(predictions, Y)
  return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, iterations, alpha):
  """
  loop that helps running forward and backpropagation with updated weights and biases
  and then printing the accuracy
  """
  W1, b1, W2, b2 = init_parameters()
  for i in range (iterations):
    A1, Z1, A2, Z2 = forward_propagation(W1, b1, W2, b2, X)
    dW1, db1, dW2, db2 = back_propagation (A1, Z1, A2, Z2, W2, X, Y)
    W1, b1, W2, b2 = update_parameters(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
    if (i % 10 == 0):
      print("Iteration: ", i)
      print("Accuracy: ", get_accuracy(get_predictions (A2), Y))
  return W1, b1, W2, b2

In [30]:
# running the actually gradient descent with the training set
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 1000, 0.1)

Iteration:  0
[5 0 0 ... 0 9 0] [8 3 7 ... 7 7 1]
Accuracy:  0.10328985507246377
Iteration:  10
[5 1 4 ... 0 6 0] [8 3 7 ... 7 7 1]
Accuracy:  0.20465217391304347
Iteration:  20
[5 1 4 ... 0 4 0] [8 3 7 ... 7 7 1]
Accuracy:  0.2951884057971014
Iteration:  30
[5 1 7 ... 1 4 9] [8 3 7 ... 7 7 1]
Accuracy:  0.36768115942028984
Iteration:  40
[5 1 7 ... 1 4 1] [8 3 7 ... 7 7 1]
Accuracy:  0.4291449275362319
Iteration:  50
[5 1 7 ... 1 4 1] [8 3 7 ... 7 7 1]
Accuracy:  0.4811884057971014
Iteration:  60
[5 1 7 ... 1 4 1] [8 3 7 ... 7 7 1]
Accuracy:  0.5239130434782608
Iteration:  70
[5 3 7 ... 1 4 1] [8 3 7 ... 7 7 1]
Accuracy:  0.5634782608695652
Iteration:  80
[4 3 7 ... 1 4 1] [8 3 7 ... 7 7 1]
Accuracy:  0.5983333333333334
Iteration:  90
[4 3 7 ... 1 4 1] [8 3 7 ... 7 7 1]
Accuracy:  0.627768115942029
Iteration:  100
[4 3 7 ... 1 7 1] [8 3 7 ... 7 7 1]
Accuracy:  0.6520869565217391
Iteration:  110
[4 3 7 ... 1 7 1] [8 3 7 ... 7 7 1]
Accuracy:  0.6733333333333333
Iteration:  120
[4 3 7 ..

In [25]:
# using the model on the testing set
Z1, A1, Z2, A2_dev = forward_propagation(W1, b1, W2, b2, X_dev)
dev_predictions = get_predictions(A2_dev)
dev_accuracy = get_accuracy(dev_predictions, Y_dev)
print("Test Accuracy:", dev_accuracy)

[4 5 5 3 5 3 5 2 2 8 1 3 2 1 4 6 5 1 0 4 6 3 5 0 2 1 0 0 5 5 5 0 2 6 6 2 7
 5 1 1 1 4 0 0 6 8 9 0 8 7 2 2 3 7 4 0 6 4 4 0 1 5 1 5 3 4 4 5 1 0 1 7 4 6
 7 1 0 2 6 6 8 1 1 3 6 1 2 6 0 1 0 3 8 6 6 9 3 1 9 5 5 8 3 5 6 4 8 1 9 3 9
 1 0 2 8 7 4 2 1 3 4 4 2 4 2 8 6 4 5 0 9 6 9 5 3 9 2 1 8 4 1 8 8 1 9 2 6 6
 2 3 7 1 6 3 1 2 2 0 5 6 0 2 9 2 0 2 6 9 1 1 7 8 8 5 4 2 1 3 4 7 3 9 0 6 0
 3 0 9 1 7 5 9 3 1 2 7 8 0 4 1 2 7 1 3 8 3 3 0 2 0 6 8 9 1 8 3 4 0 9 6 9 2
 4 4 7 1 5 0 5 3 2 4 1 4 9 3 6 6 0 8 4 4 1 4 0 1 6 4 1 7 6 0 7 7 8 1 9 2 9
 9 9 1 3 5 7 8 2 2 9 9 9 4 2 6 0 5 0 3 9 1 7 5 0 2 4 8 0 3 6 2 6 6 3 4 7 0
 8 9 8 7 5 4 2 8 6 2 3 8 1 7 0 9 7 8 4 2 8 4 4 1 2 1 6 7 2 8 7 9 8 9 3 0 2
 8 6 0 6 4 2 4 9 1 9 5 3 2 5 1 8 3 4 6 4 7 6 4 0 7 8 9 3 4 5 1 4 1 0 1 5 5
 5 3 0 7 3 0 2 9 6 4 2 3 9 9 6 8 6 3 0 0 0 2 7 5 7 1 9 6 5 1 6 0 0 9 7 2 9
 9 3 2 9 1 8 7 7 0 0 2 3 6 3 1 5 9 6 6 2 1 7 0 1 2 6 7 1 2 1 1 4 3 9 0 5 3
 8 0 2 0 3 6 2 3 4 2 3 8 3 4 5 9 9 4 3 5 1 5 0 1 1 3 5 0 6 4 2 1 9 0 7 0 0
 0 4 2 2 6 0 1 1 1 7 9 0 